# Project - AI for Medical Diagnosis and Prediction | Week #5

In this notebook, we continue our analysis of the MIMIC-CXR dataset by analyzing the performance of the CNN trained during the previous week, and especially exploring potential sources of unfairness and biases. The objective is to perform an analysis of the model and implement a strategy that could mitigate unfairness.  

We will use a subset of the **MIMIC-CXR dataset** **[1][2]**. The MIMIC Chest X-ray (MIMIC-CXR) Database v2.0.0 is a large, publicly available dataset of chest radiographs in DICOM format, accompanied by free-text radiology reports. It contains 377,110 images from 227,835 radiographic studies conducted at the Beth Israel Deaconess Medical Center in Boston, MA. The dataset has been de-identified in compliance with the US Health Insurance Portability and Accountability Act of 1996 (HIPAA) Safe Harbor requirements. All protected health information (PHI) has been removed. More details: [https://mimic.mit.edu/docs/iv/modules/cxr/](https://mimic.mit.edu/docs/iv/modules/cxr/)

<div class="alert alert-block alert-info">
<b>Your tasks are the following:</b>  <br>
- Load the dataset using the same train, validation and test splits as during the previous week <i>(Task 1)</i> <br>
- Observe the training set and the different population subgroups: gender, ethnicity, age groups <i>(Task 1*)</i> <br>
- Load the pretrained convolutional neural network <i>(Task 2)</i> <br>
- Evaluate and compare the performance of the models using appropriate metrics across subgroups <i>(Task 2*)</i> <br>
- Implement a strategy to mitigate unfairness: subsampling or data augmentation <i>(Task 3)</i> <br>
- Re-evaluate the performance of the newly trained model regarding fairness and biases <i>(Task 4)</i>
</div>

**[1]** Johnson, A., Pollard, T., Mark, R., Berkowitz, S., & Horng, S. (2024). MIMIC-CXR Database (version 2.1.0). PhysioNet. [https://doi.org/10.13026/4jqj-jw95](https://doi.org/10.13026/4jqj-jw95).

**[2]** Johnson, A.E.W., Pollard, T.J., Berkowitz, S.J. et al. MIMIC-CXR, a de-identified publicly available database of chest radiographs with free-text reports. Sci Data 6, 317 (2019). [https://doi.org/10.1038/s41597-019-0322-0](https://doi.org/10.1038/s41597-019-0322-0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

!pip install pydicom
!python3.8 -m pip install opencv-python
import pydicom
import time
import cv2
from PIL import Image
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import tv_tensors
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score

torch.mps.empty_cache()

In [ ]:
DATA_PATH = '../data/MIMIC-CXR'

If you do not have the dataset anymore, please re-run the following cell to download it. Here, we will use the chest X-ray images and the labels extracted. 

In [ ]:
# !wget https://uni-bonn.sciebo.de/s/Rb66iDHGPrJiRAq/download --output {DATA_PATH}

In [ ]:
labels_df = pd.read_csv('labels.csv')

In [ ]:
labels_df.head()

In [ ]:
test_df = pd.read_csv('test_labels.csv')
train_df = pd.read_csv('train_labels.csv')

## Task 1 - Dataset exploration 

* Explore the different demographics available in the dataset and count the number of samples in each subgroup
* Plot a few statistics about the composition of the train set, and reflect on the potential impact on fairness

## Task 2 - Performance across subgroups 

* Compute the performance of the classifier trained during week 4 on each subgroup individually
* Compare these performance across subgroups, and look for any unfavored subgroups
* Reflect and comment on the link between the performance and the composition of the train dataset

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, dataframe, image_dir, train=False):
        super().__init__()
        self.df = dataframe
        self.image_ids = self.df['dicom_id'].tolist()
        self.study_ids = self.df['study_id'].tolist()
        self.subject_ids = self.df['subject_id'].tolist()
        self.labels = self.df['pathology'].tolist()
        self.image_dir = image_dir
        self.size = 512

        if train:
            self.transforms_img = torchvision.transforms.Compose([
                torchvision.transforms.Resize((self.size, self.size)),
                torchvision.transforms.RandomAdjustSharpness(sharpness_factor=2),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((0.5,), (0.2,))
            ])
        else: 
            self.transforms_img = torchvision.transforms.Compose([
                torchvision.transforms.Resize((self.size, self.size)),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((0.5,), (0.2,))
            ])

    def __getitem__(self, index: int):
        image_id = self.image_ids[index]
        study_id = self.study_ids[index]
        subject_id = self.subject_ids[index]
        target = self.labels[index]
        
        image_path = f'{self.image_dir}/p{subject_id}/s{study_id}/{image_id}.dcm'
        image_dicom = pydicom.dcmread(image_path)
        image_array = image_dicom.pixel_array
        image_array = (image_array / image_array.max() * 255).astype(np.uint8)
        image = Image.fromarray(image_array, mode = "L")
        image = self.transforms_img(image)

        target = torch.tensor(target)

        return image, target

    def __len__(self) -> int:
        return len(self.image_ids)

In [ ]:
model_enb0 = efficientnet_b0()
model_enb0.classifier[1] = nn.Linear(1280, 2)
model_enb0.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
model_enb0.load_state_dict(torch.load('./efficientnetb0-pathology-cxr.pth'))

test_df = pd.read_csv('./test_labels.csv')

test_dataset_cls = ClassificationDataset(
    test_df, 
    f'{DATA_PATH}/files/p18',
    train = False
)

test_dataloader = DataLoader(test_dataset_cls, batch_size=1)

model_enb0.eval()
model_enb0.to(device)

preds_list = []
probs_list = []
label_list = []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        out = model_enb0(X)

        probs = nn.functional.softmax(out, dim=1)  # use softmax for multi-logit outputs
        preds = probs.argmax(dim=1)

        probs_list.append(probs[:, 1].item())  # probability for class 1
        preds_list.append(preds.item())
        label_list.append(y.detach().cpu().numpy())

display = RocCurveDisplay.from_predictions(
label_list,
probs_list,
name=f"Pneumonia vs. rest",
color="darkorange",
plot_chance_level=True,
)
_ = display.ax_.set(
xlabel="False Positive Rate",
ylabel="True Positive Rate",
title="Binary classification",
)
display.ax_.legend(frameon=False)

print(
f'Model EN-B0:',
'\nAccuracy:', accuracy_score(label_list, preds_list),
'\nF1-score:', f1_score(label_list, preds_list),
'\nRecall:', recall_score(label_list, preds_list),
'\nPrecision:', precision_score(label_list, preds_list)
)